# 🚀 Build Your Own Custom LLM From Scratch (Zero-Dependency)
### Complete Pipeline: Custom BPE Tokenizer -> 32.5M Transformer -> Pre-Training -> WhatsApp SFT -> Interactive Chat

> **🛡️ Guarantee:** Zero pre-existing weights (no LLaMA/Mistral/Qwen download), Zero external APIs (no OpenAI/Gemini). 100% created and trained locally on your compute.

## Step 1: Install Dependencies & Check Device Acceleration

In [ ]:
# Install dependencies (uncomment if running on fresh system or Colab)
# !pip install -q torch transformers tokenizers datasets trl accelerate

import torch

if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print(f"Active Hardware Device: {device.upper()}")
if device == "cuda":
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")

## Step 2: Extract Raw Pre-Training Text Corpus from WhatsApp Dataset

In [ ]:
import os, csv, time
from pathlib import Path

csv_path = "01_data/whatsapp_chat_14lakh.csv"
corpus_path = "01_data/raw_pretrain_corpus.txt"

print("Extracting raw messages for pre-training...")
count = 0
start_t = time.time()

with open(csv_path, "r", encoding="utf-8") as f_in, \
     open(corpus_path, "w", encoding="utf-8", buffering=2*1024*1024) as f_out:
    reader = csv.DictReader(f_in)
    for row in reader:
        msg = row.get("message", "").strip()
        if msg and len(msg) > 10:
            f_out.write(msg + "\n")
            count += 1

print(f"Done in {time.time() - start_t:.2f}s! Extracted {count:,} lines to {corpus_path}")

## Step 3: Train Custom Byte-Level BPE Tokenizer from Scratch

In [ ]:
from tokenizers import ByteLevelBPETokenizer
from transformers import PreTrainedTokenizerFast

tok_dir = "02_tokenizer/model"
os.makedirs(tok_dir, exist_ok=True)

print("Training Byte-Level BPE Tokenizer (Vocab Size: 16,384)...")
tokenizer = ByteLevelBPETokenizer()
tokenizer.train(
    files=[corpus_path],
    vocab_size=16384,
    min_frequency=2,
    special_tokens=["<s>", "<pad>", "</s>", "<unk>", "<|im_start|>", "<|im_end|>"]
)
tokenizer.save_model(tok_dir)

# Wrap into Hugging Face interface
hf_tokenizer = PreTrainedTokenizerFast(
    tokenizer_file=f"{tok_dir}/vocab.json",
    bos_token="<s>", eos_token="</s>", pad_token="<pad>", unk_token="<unk>"
)
print("Tokenizer ready! Vocab size:", len(hf_tokenizer))

# Test encoding
sample = "Are you free for coffee this evening?"
tokens = hf_tokenizer(sample)
print("Sample:", sample)
print("Token IDs:", tokens["input_ids"])
print("Decoded back:", hf_tokenizer.decode(tokens["input_ids"]))

## Step 4: Define Blank 32.5M Parameter Decoder-Only Transformer Architecture

In [ ]:
from transformers import LlamaConfig, LlamaForCausalLM

# Balanced architecture optimized for local RAM & VRAM execution
config = LlamaConfig(
    vocab_size=len(hf_tokenizer),
    hidden_size=512,               # Token vector dimension
    intermediate_size=1376,        # SwiGLU FFN dimension (~2.7x)
    num_hidden_layers=8,           # 8 Transformer Blocks
    num_attention_heads=8,         # 8 Attention Heads
    max_position_embeddings=512,   # Context window
    pad_token_id=1, bos_token_id=0, eos_token_id=2
)

# Initialize completely blank model with random Gaussian weights
model = LlamaForCausalLM(config)
total_params = model.num_parameters()
print(f"Initialized blank LLM. Trainable parameters: {total_params:,} (~{total_params/1e6:.1f}M)")

## Step 5: Stage 1 Pre-Training (Next-Token Prediction on Raw Text)

In [ ]:
import math
from datasets import load_dataset
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

base_model_dir = "04_pretrain/base_model"
os.makedirs(base_model_dir, exist_ok=True)

# Load raw text dataset
raw_ds = load_dataset("text", data_files={"train": corpus_path})

def tokenize_fn(batch):
    return hf_tokenizer(batch["text"], truncation=True, max_length=512, padding="max_length")

tokenized_ds = raw_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
collator = DataCollatorForLanguageModeling(tokenizer=hf_tokenizer, mlm=False)

train_args = TrainingArguments(
    output_dir=base_model_dir,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-4,
    lr_scheduler_type="cosine",
    warmup_steps=100,
    num_train_epochs=3,
    logging_steps=20,
    save_strategy="epoch",
    fp16=torch.cuda.is_available(),
    use_mps_device=(device == "mps"),
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=train_args,
    data_collator=collator,
    train_dataset=tokenized_ds["train"]
)

print("Starting Pre-Training (Learning language from scratch)...")
train_res = trainer.train()

# Perplexity metric
loss = train_res.training_loss
ppl = math.exp(loss) if loss < 20 else float("inf")
print(f"Pre-Training Done! Loss: {loss:.4f} | Perplexity: {ppl:.2f}")

# Save Base Model Checkpoint
model.save_pretrained(base_model_dir)
hf_tokenizer.save_pretrained(base_model_dir)

## Step 6: Stage 2 Supervised Fine-Tuning (SFT) with 7 Lakh WhatsApp ChatML Dialogues

In [ ]:
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM

chat_model_dir = "05_finetune/chat_model"
os.makedirs(chat_model_dir, exist_ok=True)

# Load conversational SFT dataset
chat_ds = load_dataset("json", data_files={
    "train": "01_data/train_whatsapp_700k.jsonl",
    "validation": "01_data/val_whatsapp_20k.jsonl"
})

# Mask user questions so the model ONLY computes loss on assistant replies
response_collator = DataCollatorForCompletionOnlyLM(
    response_template="<|im_start|>assistant\n",
    tokenizer=hf_tokenizer
)

sft_args = TrainingArguments(
    output_dir=chat_model_dir,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=20,
    save_strategy="epoch",
    evaluation_strategy="epoch",
    fp16=torch.cuda.is_available(),
    use_mps_device=(device == "mps"),
    report_to="none"
)

sft_trainer = SFTTrainer(
    model=model,
    train_dataset=chat_ds["train"],
    eval_dataset=chat_ds["validation"],
    dataset_text_field="text",
    data_collator=response_collator,
    max_seq_length=512,
    tokenizer=hf_tokenizer,
    args=sft_args
)

print("Starting Supervised Fine-Tuning (SFT)... ")
sft_trainer.train()

# Save Final Chatbot Weights
sft_trainer.model.save_pretrained(chat_model_dir)
hf_tokenizer.save_pretrained(chat_model_dir)
print(f"Chatbot LLM successfully saved to {chat_model_dir}!")

## Step 7: Live Interactive Chat (Test Your Custom LLM)

In [ ]:
def chat_with_model(prompt, max_tokens=80, temperature=0.7):
    formatted = f"<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"
    inputs = hf_tokenizer(formatted, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids,
            max_new_tokens=max_tokens,
            temperature=temperature,
            top_p=0.9,
            repetition_penalty=1.15,
            do_sample=True,
            pad_token_id=hf_tokenizer.pad_token_id or hf_tokenizer.eos_token_id
        )
    
    reply = hf_tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True)
    return reply.split("<|im_end|>")[0].strip()

# Test Queries
test_prompts = [
    "Hey! Are you free for coffee this evening?",
    "What would you recommend for lunch today?",
    "How is the monthly report going?"
]

for q in test_prompts:
    print(f"User: {q}")
    print(f"Custom LLM: {chat_with_model(q)}\n" + "-"*50)